In [0]:
# All Instances
df = spark.table("`ph-com-ai-us-prod-catalog-2553213839599380`.kerendia_oe_journey.final_mdb_oe_2_0_all_instances")

In [0]:
df.display()

In [0]:
import pandas as pd

# ================================
# STEP 1 : Check Data Type of All Columns
# ================================

dtype_df = pd.DataFrame({
    'Column_Name': [f.name for f in df.schema.fields],
    'Data_Type': [str(f.dataType) for f in df.schema.fields],
})

print(f"Total columns: {len(dtype_df)}")
display(dtype_df)

In [0]:
# Count number of features per datatype
dtype_counts = dtype_df.groupby('Data_Type').size().reset_index(name='Feature_Count')
display(dtype_counts)

In [0]:
from pyspark.sql import functions as F

# ================================
# STEP 2 : Check Null Values Across All Features
# ================================

total_rows = df.count()

# Count nulls per column using a single select (efficient for wide DataFrames)
null_counts = df.select(
    [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]
).toPandas().T.reset_index()

null_counts.columns = ['Feature', 'Null_Count']
null_counts['Null_Percent'] = round((null_counts['Null_Count'] / total_rows) * 100, 2)
null_counts = null_counts.sort_values('Null_Percent', ascending=False).reset_index(drop=True)

# Summary
features_with_nulls = null_counts[null_counts['Null_Count'] > 0]
print(f"Total features: {len(null_counts)}")
print(f"Features WITH null values: {len(features_with_nulls)}")
print(f"Features WITHOUT null values: {len(null_counts) - len(features_with_nulls)}")
print()
display(null_counts)

In [0]:
df.select([F.approx_count_distinct(F.col(c)).alias(c) for c in df.columns]).display()

In [0]:
(df.count(), len(df.columns))